# Setup and load data

Imports

In [27]:
import pandas as pd
from pathlib import Path
import numpy as np
import os

Merge csv

In [28]:
DATA_DIR = Path("./dataset_raw")
files = sorted(DATA_DIR.glob("*.csv"))

print("Files found:")
for f in files:
    print(f)


dfs = []

for f in files:
    print(f"Loading {f.name}")
    df = pd.read_csv(
        f,
        sep=",",
        low_memory=False
    )
    dfs.append(df)

dvf = pd.concat(dfs, ignore_index=True)

In [29]:
print("Total rows:", len(dvf))
dvf.head()

Total rows: 326804


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2021-1289503,2021-01-05,1,Vente,352000.0,228.0,NaN,RUE DE L EGLISE,0220,77115.0,...,Maison,120.0,3.0,S,sols,NaN,NaN,331.0,2.755044,48.527196
1,2021-1289503,2021-01-05,1,Vente,352000.0,228.0,NaN,RUE DE L EGLISE,0220,77115.0,...,Maison,120.0,3.0,S,sols,NaN,NaN,45.0,2.755044,48.527196
2,2021-1289504,2021-01-04,1,Vente,6401564.0,NaN,NaN,ESSEAUNE,B013,77550.0,...,NaN,NaN,NaN,T,terres,NaN,NaN,25203.0,2.607463,48.610680
3,2021-1289504,2021-01-04,1,Vente,6401564.0,NaN,NaN,ESSEAUNE,B013,77550.0,...,NaN,NaN,NaN,T,terres,NaN,NaN,402.0,2.606522,48.613654
4,2021-1289504,2021-01-04,1,Vente,6401564.0,NaN,NaN,ESSEAUNE,B013,77550.0,...,NaN,NaN,NaN,T,terres,NaN,NaN,1497.0,2.606341,48.613514


In [30]:
print("Columns:")
print(dvf.columns.tolist())

## 4.1 filtering

In [31]:
# Keep only sales
dvf = dvf[dvf["nature_mutation"] == "Vente"]

# Keep only the property types we want
dvf = dvf[dvf["type_local"].isin(["Maison", "Appartement"])]

# Optional: remove non-sale artifacts like very small price
dvf = dvf[dvf["valeur_fonciere"] > 1000] 

## 4.2

In [32]:
# Rules: houses need surface_terrain, apartments do not
mask_maison = dvf["type_local"] == "Maison"
mask_appart = dvf["type_local"] == "Appartement"

# Drop rows with missing critical columns
dvf = dvf[~((mask_maison) & 
            dvf[["valeur_fonciere","surface_reelle_bati","nombre_pieces_principales","surface_terrain"]].isna().any(axis=1))]
dvf = dvf[~((mask_appart) & 
            dvf[["valeur_fonciere","surface_reelle_bati","nombre_pieces_principales"]].isna().any(axis=1))]

# Remove unrealistic zeros
dvf = dvf[~((mask_maison) & ((dvf["surface_reelle_bati"] <= 10) | 
                              (dvf["surface_terrain"] <= 10) | 
                              (dvf["nombre_pieces_principales"] <= 0)))]
dvf = dvf[~((mask_appart) & ((dvf["surface_reelle_bati"] <= 5) | 
                              (dvf["nombre_pieces_principales"] <= 0)))]

C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1619730057.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dvf = dvf[~((mask_appart) &
C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1619730057.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dvf = dvf[~((mask_maison) & ((dvf["surface_reelle_bati"] <= 10) |
C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1619730057.py:15: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dvf = dvf[~((mask_appart) & ((dvf["surface_reelle_bati"] <= 5) |


## 4.3

In [33]:
num_cols_maison = ["valeur_fonciere","surface_reelle_bati","surface_terrain","nombre_pieces_principales"]
num_cols_appart = ["valeur_fonciere","surface_reelle_bati","nombre_pieces_principales"]

# Function to trim
def trim_outliers(df, mask, cols):
    for col in cols:
        lower = df.loc[mask, col].quantile(0.025)
        upper = df.loc[mask, col].quantile(0.975)
        df = df[~(mask & ~df[col].between(lower, upper))]
        print(f"{col} ({'Maison' if mask.any() else 'Appartement'}): keeping {lower:.0f} – {upper:.0f}")
    return df

dvf = trim_outliers(dvf, mask_maison, num_cols_maison)
dvf = trim_outliers(dvf, mask_appart, num_cols_appart)

C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1988533941.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1988533941.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1988533941.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1988533941.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]


valeur_fonciere (Maison): keeping 85000 – 900000
surface_reelle_bati (Maison): keeping 40 – 210
surface_terrain (Maison): keeping 55 – 1750
nombre_pieces_principales (Maison): keeping 2 – 7
valeur_fonciere (Maison): keeping 66000 – 13465398
surface_reelle_bati (Maison): keeping 19 – 103
nombre_pieces_principales (Maison): keeping 1 – 5


C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1988533941.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1988533941.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\alexi\AppData\Local\Temp\ipykernel_26312\1988533941.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]


# 4.4

In [34]:
lot_cols_surface = ["lot1_surface_carrez","lot2_surface_carrez","lot3_surface_carrez","lot4_surface_carrez","lot5_surface_carrez"]
lot_cols_num = ["lot1_numero","lot2_numero","lot3_numero","lot4_numero","lot5_numero"]

dvf.loc[mask_appart, "total_carrez_surface"] = dvf.loc[mask_appart, lot_cols_surface].sum(axis=1, skipna=True)
dvf.loc[mask_appart, "number_of_lots"] = dvf.loc[mask_appart, lot_cols_num].notna().sum(axis=1)

# 4.5 INSEE

In [35]:
# Load the INSEE indices
TMP_DIR = "./tmp_downloads"

insee_path = os.path.join(TMP_DIR, "valeurs_trimestrielles.csv")
insee_df = pd.read_csv(insee_path, sep=";")
insee_df = insee_df.dropna(subset=["Codes"])
insee_df = insee_df.drop("Codes", axis=1)
insee_df = insee_df.rename(columns={"Libellé": "Year-Q", "Indice des prix des logements (neufs et anciens) – Brut – Base 100 en moyenne annuelle 2015": "indice"})
insee_df = insee_df.reset_index(drop=True)
# Inspect the data
insee_df.head()

,Year-Q,indice
0,2025-T3,128.5
1,2025-T2,126.5
2,2025-T1,126.5
3,2024-T4,126.3
4,2024-T3,127.6


In [36]:
# Split 'Year-Q' into 'year' and 'quarter'
insee_df[["year", "quarter"]] = insee_df["Year-Q"].str.split("-T", expand=True)
insee_df["year"] = insee_df["year"].astype(int)
insee_df["quarter"] = insee_df["quarter"].astype(int)

# Convert 'indice' to numeric (float)
insee_df["indice"] = pd.to_numeric(insee_df["indice"], errors="coerce")

# Get the lowest available year from the data (instead of hardcoding 2020)
base_year = insee_df["year"].min()

# Ensure that the base_year exists in the dataset
base_index_row = insee_df[(insee_df["year"] == base_year) & (insee_df["quarter"] == 1)]

# Check if the row for base_year-T1 exists
base_index = base_index_row["indice"].values[0]  # base_index is a float
# Normalize the indices to the base year and quarter (e.g., base_year-T1 = 100)
insee_df["normalized_index"] = insee_df["indice"] / base_index * 100
insee_df["normalized_index"] = insee_df["normalized_index"].round().astype(int)

# Inspect the updated INSEE DataFrame after splitting and normalization
insee_df.head()

,Year-Q,indice,year,quarter,normalized_index
0,2025-T3,128.5,2025,3,107
1,2025-T2,126.5,2025,2,105
2,2025-T1,126.5,2025,1,105
3,2024-T4,126.3,2024,4,105
4,2024-T3,127.6,2024,3,106


In [37]:
# Convert 'date_mutation' to datetime if it's not already
dvf["date_mutation"] = pd.to_datetime(dvf["date_mutation"], errors="coerce")

# Extract 'year' and 'quarter' from 'date_mutation' column in DVF data
dvf["year"] = dvf["date_mutation"].dt.year
dvf["quarter"] = dvf["date_mutation"].dt.quarter

# Merge INSEE data into DVF data based on 'year' and 'quarter'
dvf = dvf.merge(insee_df[["year", "quarter", "normalized_index"]], on=["year", "quarter"], how="left")

# Inspect the merged data
dvf["valeur_fonciere_actualisee"] = np.floor(dvf["valeur_fonciere"] * (dvf["normalized_index"] / 100))
dvf = dvf.drop(columns=["year", "quarter", "normalized_index"])

print(dvf[["valeur_fonciere","valeur_fonciere_actualisee"]].head())
# dvf.to_csv("datasets_prepd/dvf_prepared.csv", index=False)

# 4.6 M²

In [38]:
# Compute price per m² for each transaction (ignore NaN values for surface_reelle_bati)
dvf["prix_m2"] = dvf["valeur_fonciere"] / dvf["surface_reelle_bati"]

# Calculate the average price per m² by postal code (code_postal)
dvf["prix_m2_ref"] = dvf.groupby("code_postal")["prix_m2"].transform("mean")

# Inspect the first few rows to verify the new column 'prix_m2_ref'
print(dvf[["code_postal", "valeur_fonciere", "surface_reelle_bati", "prix_m2", "prix_m2_ref"]].head())

# Split datasets

In [39]:
# Filter the dataset into Maison and Appartement
maison = dvf[dvf["type_local"] == "Maison"]
appart = dvf[dvf["type_local"] == "Appartement"]

os.makedirs("datasets_prepd", exist_ok=True)

maison.to_csv("datasets_prepd/dvf_maison.csv", index=False)
appart.to_csv("datasets_prepd/dvf_appartement.csv", index=False)